# Administrative geographic names

Explore the published GeoParquet: its columns, data types, and administrative hierarchy in CoDPA code order.

Use a Python environment with `geopandas` and `pyarrow` installed. Run the cells from top to bottom, with the working directory set to this repository or its `examples/` folder. The notebook only reads the data.


In [41]:
from pathlib import Path
import geopandas as gpd

ROOT = Path.cwd()
if ROOT.name == "examples":
    ROOT = ROOT.parent

ADMIN_NAMES = ROOT / "data/administrative-geographic-names/cuba_administrative_geographic_names.parquet"
names = gpd.read_parquet(ADMIN_NAMES)

print(f"Rows: {len(names):,}")
print(f"Columns: {len(names.columns)}")
print(f"CRS: {names.crs.to_string()}")

Rows: 183
Columns: 12
CRS: EPSG:4326


## Columns and data types

These are the column names and data types loaded by GeoPandas. CoDPA codes are kept as text, preserving codes such as `21.01` and `21.10`.


In [42]:
structure = names.dtypes.astype(str).rename_axis("column").reset_index(name="data_type")
print(structure.to_string(index=False))

       column data_type
           id    string
   codpa_code       str
         name       str
        level       str
province_code       str
province_name       str
       gns_id       str
  geonames_id       str
  overture_id       str
          lat   float64
          lon   float64
     geometry  geometry


## First records


In [43]:
print(names.head().to_string(index=False))

                                  id codpa_code                name        level province_code province_name   gns_id geonames_id                          overture_id       lat        lon                   geometry
01a0b893-9647-773c-87f5-2e70378ee93d         21       Pinar del Río     province            21 Pinar del Río -1638165     3544088 7c040c50-ff5f-485c-86e3-e9d4f2ff776f 22.416670 -83.833330 POINT (-83.83333 22.41667)
01a0b893-9647-773c-87f5-2e71a71b0886      21.01             Sandino municipality            21 Pinar del Río 14592999    11287605 c8dce969-52e3-4195-903d-73c308a2850e 22.042999 -84.207086   POINT (-84.20709 22.043)
01a0b893-9647-773c-87f5-2e720004ac12      21.02              Mantua municipality            21 Pinar del Río -1634314     3547929 56dbd045-8c39-41a9-9bc2-51a7c2dc0bce 22.339615 -84.259327 POINT (-84.25933 22.33962)
01a0b893-9647-773c-87f5-2e731a7aea47      21.03 Minas de Matahambre municipality            21 Pinar del Río 14593000    11288064 4567612c-7

## CoDPA hierarchy

Each province is followed by its municipalities, ordered by CoDPA code and indented beneath the province. Parent relationships come from `province_code`.

Isla de la Juventud is listed separately as a special municipality with its existing code `40.01`. It has no parent province in this table; no additional province record is created.


In [44]:
provinces = names.loc[names["level"].eq("province")].sort_values("codpa_code")
municipalities = names.loc[names["level"].eq("municipality")]
special_municipalities = names.loc[names["level"].eq("special_municipality")].sort_values("codpa_code")

print("PROVINCIAS Y MUNICIPIOS\n")
for province in provinces.itertuples(index=False):
    print(f"{province.codpa_code}  {province.name}")
    children = municipalities.loc[
        municipalities["province_code"].eq(province.codpa_code)
    ].sort_values("codpa_code")
    for municipality in children.itertuples(index=False):
        print(f"\t{municipality.codpa_code}  {municipality.name}")
    print()

print("MUNICIPIO ESPECIAL\n")
for municipality in special_municipalities.itertuples(index=False):
    print(f"\t{municipality.codpa_code}  {municipality.name}")

PROVINCIAS Y MUNICIPIOS

21  Pinar del Río
	21.01  Sandino
	21.02  Mantua
	21.03  Minas de Matahambre
	21.04  Viñales
	21.05  La Palma
	21.06  Los Palacios
	21.07  Consolación del Sur
	21.08  Pinar del Río
	21.09  San Luis
	21.10  San Juan y Martínez
	21.11  Guane

22  Artemisa
	22.01  Bahía Honda
	22.02  Mariel
	22.03  Guanajay
	22.04  Caimito
	22.05  Bauta
	22.06  San Antonio de los Baños
	22.07  Güira de Melena
	22.08  Alquízar
	22.09  Artemisa
	22.10  Candelaria
	22.11  San Cristóbal

23  La Habana
	23.01  Playa
	23.02  Plaza de la Revolución
	23.03  Centro Habana
	23.04  La Habana Vieja
	23.05  Regla
	23.06  La Habana del Este
	23.07  Guanabacoa
	23.08  San Miguel del Padrón
	23.09  Diez de Octubre
	23.10  Cerro
	23.11  Marianao
	23.12  La Lisa
	23.13  Boyeros
	23.14  Arroyo Naranjo
	23.15  Cotorro

24  Mayabeque
	24.01  Bejucal
	24.02  San José de las Lajas
	24.03  Jaruco
	24.04  Santa Cruz del Norte
	24.05  Madruga
	24.06  Nueva Paz
	24.07  San Nicolás
	24.08  Güines
	24.09  Mel